# IPL Player Detection — Feature Extraction
**Group 42 | IITB PML Sem 1**

This notebook extracts all visual features for each 100×75 px grid cell across every image and saves the result to `pipeline/features/ps1/cell_features.csv`.

**Output:** one row per cell, 63,488 rows (992 images × 64 cells), 274 columns

| Feature group | Columns | Description |
|---|---|---|
| HSV/RGB histograms | `f000`–`f191` (192) | 64-bin histograms for Hue, Saturation, Value channels |
| Texture (LBP + GLCM) | `tex_00`–`tex_25` (26) | 10 LBP + 16 GLCM (contrast/homogeneity/energy/correlation × 4 angles) |
| Edge | `f_edge_density`, `f_canny_count`, `f_sobel_mean`, `f_sobel_std`, `f_laplacian_var`, `f_contour_count` (6) | Canny, Sobel, Laplacian statistics |
| HOG orientation | `f_hog_000`–`f_hog_035` (36) | 4 cells-per-block × 9-orientation HOG descriptor |
| HOG 9-bin | `hog_bin_0`–`hog_bin_8` (9) | Single 9-bin HOG for the whole cell |

**Images are downloaded from HuggingFace** on first run (cached locally).
**Annotations CSV** (`Dataset_Annotations.csv`) must be present in `pipeline/labels/`.

## 1. Imports & Configuration

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import cv2
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern, hog
from skimage.color import rgb2gray
from huggingface_hub import snapshot_download

# ── Config ────────────────────────────────────────────────────────────────────
HF_REPO_ID       = 'goyaljai/IPL-Player-Detection-IITB-PML'
ANNOTATIONS_CSV  = Path('..') / 'pipeline' / 'labels' / 'Dataset_Annotations.csv'
OUTPUT_CSV       = Path('..') / 'pipeline' / 'features' / 'ps1' / 'cell_features.csv'

# Images are 800×600, grid is 8×8
IMG_W, IMG_H = 800, 600
GRID_ROWS, GRID_COLS = 8, 8
CELL_W = IMG_W // GRID_COLS   # 100
CELL_H = IMG_H // GRID_ROWS   # 75

# Images to exclude (confirmed bad quality in §2b of pipeline notebook)
EXCLUDED_IMAGES = {
    'img_1006.jpg','img_1007.jpg','img_1008.jpg','img_1009.jpg',
    'img_1010.jpg','img_1011.jpg','img_1012.jpg','img_1013.jpg',
    'img_89.jpg','img_144.jpg','img_201.jpg','img_267.jpg',
    'img_271.jpg','img_276.jpg','img_281.jpg','img_402.jpg',
    'img_423.jpg','img_454.jpg','img_517.jpg','img_661.jpg','img_753.jpg',
}

print(f'Output CSV   : {OUTPUT_CSV}')
print(f'Cell size    : {CELL_W}×{CELL_H} px')
print(f'Excluded     : {len(EXCLUDED_IMAGES)} images')

## 2. Load Annotations & Download Images

In [ ]:
# ── Load annotations ──────────────────────────────────────────────────────────
annotations = pd.read_csv(ANNOTATIONS_CSV)

# Remove excluded images
annotations = annotations[~annotations['Image File Name'].isin(EXCLUDED_IMAGES)].reset_index(drop=True)

print(f'Annotations  : {len(annotations)} images after exclusions')
print(f'Train / Test : {(annotations["Train Or Test"]=="Train").sum()} / {(annotations["Train Or Test"]=="Test").sum()}')

# ── Download dataset from HuggingFace ─────────────────────────────────────────
print('\nDownloading/verifying dataset from HuggingFace (cached after first run)...')
dataset_dir = Path(snapshot_download(repo_id=HF_REPO_ID, repo_type='dataset'))
print(f'Dataset dir  : {dataset_dir}')

def find_image(img_name, split):
    """Return image path; falls back to searching both train/test dirs."""
    p = dataset_dir / split.lower() / img_name
    if p.exists(): return p
    for d in ['train', 'test']:
        p = dataset_dir / d / img_name
        if p.exists(): return p
    return None

## 3. Feature Extraction Functions

One function per feature group. All operate on a single cropped cell (100×75 px PIL/numpy patch).

In [ ]:
# ── HSV/RGB histograms — 192 features (f000–f191) ─────────────────────────────
def extract_hsv_features(patch_pil):
    """
    64-bin histograms for each of the 3 HSV channels.
    Returns 192 integer counts.
    """
    hsv = np.array(patch_pil.convert('HSV'))
    h_hist, _ = np.histogram(hsv[:,:,0], bins=64, range=(0, 256))
    s_hist, _ = np.histogram(hsv[:,:,1], bins=64, range=(0, 256))
    v_hist, _ = np.histogram(hsv[:,:,2], bins=64, range=(0, 256))
    return np.concatenate([h_hist, s_hist, v_hist]).astype(int)  # 192 values

In [ ]:
# ── Texture: LBP + GLCM — 26 features (tex_00–tex_25) ────────────────────────
def extract_texture_features(patch_rgb):
    """
    10 LBP uniform histogram values + 16 GLCM properties.
    Input: (75, 100, 3) uint8 numpy array.
    Returns 26 float values.
    """
    gray = (rgb2gray(patch_rgb) * 255).astype(np.uint8)

    # LBP — 10 values
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10))
    lbp_hist = lbp_hist.astype(float) / (lbp_hist.sum() + 1e-6)  # normalised

    # GLCM — 4 properties × 4 angles = 16 values
    gray_q = (gray / 8).astype(np.uint8)  # quantise to 32 levels
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    glcm = graycomatrix(gray_q, distances=[1], angles=angles,
                        levels=32, symmetric=True, normed=True)
    glcm_feats = []
    for prop in ['contrast', 'homogeneity', 'energy', 'correlation']:
        glcm_feats.extend(graycoprops(glcm, prop).ravel())  # 4 values per prop

    return np.concatenate([lbp_hist, glcm_feats])  # 26 values

In [ ]:
# ── Edge features — 6 values ─────────────────────────────────────────────────
def compute_edge_maps(image_bgr):
    """Compute full-image edge maps once per image (reused for all 64 cells)."""
    gray    = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    canny   = cv2.Canny(blurred, threshold1=50, threshold2=150)
    sobel_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)
    laplacian = cv2.Laplacian(blurred, cv2.CV_64F)
    return {'canny': canny, 'sobel_mag': sobel_mag, 'laplacian': laplacian}

def extract_edge_features(edge_maps, x1, y1, x2, y2):
    """
    Returns 6 values:
      f_edge_density, f_canny_count, f_sobel_mean, f_sobel_std,
      f_laplacian_var, f_contour_count
    """
    canny_cell   = edge_maps['canny'][y1:y2, x1:x2]
    sobel_cell   = edge_maps['sobel_mag'][y1:y2, x1:x2]
    lap_cell     = edge_maps['laplacian'][y1:y2, x1:x2]
    area         = canny_cell.shape[0] * canny_cell.shape[1]

    canny_count  = int(np.count_nonzero(canny_cell))
    edge_density = canny_count / area if area > 0 else 0.0
    sobel_mean   = float(np.mean(sobel_cell))
    sobel_std    = float(np.std(sobel_cell))
    lap_var      = float(np.var(lap_cell))

    # Contour count from the Canny cell
    cnts, _      = cv2.findContours(canny_cell, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour_count = len(cnts)

    return [edge_density, canny_count, sobel_mean, sobel_std, lap_var, contour_count]

In [ ]:
# ── HOG features — 36 orientation + 9 bin = 45 values ────────────────────────
def extract_hog_features(patch_rgb):
    """
    Returns 45 values:
      f_hog_000–f_hog_035  : 4-cell HOG descriptor (36 values)
      hog_bin_0–hog_bin_8  : single 9-bin HOG for whole cell (9 values)
    Input: (75, 100, 3) uint8 numpy array.
    """
    gray = rgb2gray(patch_rgb)   # (75, 100) float64

    # 36-value descriptor: 2×2 blocks of cells, 9 orientations
    # pixels_per_cell=(37,50) → 2 cells across each axis → 4 cells × 9 = 36
    hog_36 = hog(
        gray,
        orientations=9,
        pixels_per_cell=(37, 50),
        cells_per_block=(1, 1),
        feature_vector=True,
        channel_axis=None
    )

    # 9-bin: treat entire cell as one block
    hog_9 = hog(
        gray,
        orientations=9,
        pixels_per_cell=(75, 100),
        cells_per_block=(1, 1),
        feature_vector=True,
        channel_axis=None
    )

    return hog_36, hog_9   # shapes (36,) and (9,)

## 4. Main Extraction Loop

Processes one image at a time. Edge maps are computed once per image and reused for all 64 cells.

In [ ]:
CELL_COLS = [f'c{i:02d}' for i in range(1, 65)]
rows      = []
missing   = []

for _, ann_row in tqdm(annotations.iterrows(), total=len(annotations), desc='Extracting features'):
    img_name  = ann_row['Image File Name']
    split     = ann_row['Train Or Test']
    img_path  = find_image(img_name, split)

    if img_path is None:
        missing.append(img_name)
        continue

    # Load image in both formats needed
    img_pil = Image.open(img_path).convert('RGB').resize((IMG_W, IMG_H))
    img_rgb = np.array(img_pil, dtype=np.uint8)                  # (600,800,3)
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)            # for OpenCV

    # Compute edge maps once per image
    edge_maps = compute_edge_maps(img_bgr)

    # Extract image_number (integer from filename, e.g. img_42.jpg → 42)
    try:
        image_number = int(''.join(filter(str.isdigit, img_name.split('.')[0])))
    except Exception:
        image_number = -1

    cell_idx = 0
    for cell_row in range(GRID_ROWS):
        for cell_col in range(GRID_COLS):
            cell_idx += 1
            x1 = cell_col * CELL_W
            y1 = cell_row * CELL_H
            x2 = x1 + CELL_W
            y2 = y1 + CELL_H

            label  = int(ann_row[f'c{cell_idx:02d}'])
            patch_pil = img_pil.crop((x1, y1, x2, y2))
            patch_rgb = img_rgb[y1:y2, x1:x2]

            # Extract all features
            hsv_feats  = extract_hsv_features(patch_pil)           # 192
            tex_feats  = extract_texture_features(patch_rgb)       # 26
            edge_feats = extract_edge_features(edge_maps, x1, y1, x2, y2)  # 6
            hog_36, hog_9 = extract_hog_features(patch_rgb)        # 36 + 9

            record = {
                'Image File Name': img_name,
                'Train Or Test'  : split,
                'cell_row'       : cell_row,
                'cell_col'       : cell_col,
                'label'          : label,
            }

            # HSV/RGB: f000–f191
            for i, v in enumerate(hsv_feats):
                record[f'f{i:03d}'] = int(v)

            # Texture: tex_00–tex_25
            for i, v in enumerate(tex_feats):
                record[f'tex_{i:02d}'] = float(v)

            # Edge: 6 named features
            names = ['f_edge_density','f_canny_count','f_sobel_mean',
                     'f_sobel_std','f_laplacian_var','f_contour_count']
            for n, v in zip(names, edge_feats):
                record[n] = v

            # HOG orientation: f_hog_000–f_hog_035
            for i, v in enumerate(hog_36):
                record[f'f_hog_{i:03d}'] = float(v)

            # HOG 9-bin: hog_bin_0–hog_bin_8
            for i, v in enumerate(hog_9):
                record[f'hog_bin_{i}'] = float(v)

            # Surrogate image number
            record['image_number'] = image_number

            rows.append(record)

print(f'\nExtracted  : {len(rows):,} rows from {len(annotations)-len(missing)} images')
if missing:
    print(f'Missing    : {len(missing)} images — {missing[:5]}')

## 5. Validate & Save

In [ ]:
df_out = pd.DataFrame(rows)

# ── Validate column count and order ──────────────────────────────────────────
expected_cols = (
    ['Image File Name', 'Train Or Test', 'cell_row', 'cell_col', 'label'] +
    [f'f{i:03d}' for i in range(192)] +
    [f'tex_{i:02d}' for i in range(26)] +
    ['f_edge_density','f_canny_count','f_sobel_mean','f_sobel_std','f_laplacian_var','f_contour_count'] +
    [f'f_hog_{i:03d}' for i in range(36)] +
    [f'hog_bin_{i}' for i in range(9)] +
    ['image_number']
)

assert list(df_out.columns) == expected_cols, \
    f'Column mismatch!\nGot:      {list(df_out.columns)}\nExpected: {expected_cols}'

print(f'Shape       : {df_out.shape}  (expected ~63,488 × 274)')
print(f'Unique imgs : {df_out["Image File Name"].nunique()}  (expected 992)')
print(f'Columns     : {len(df_out.columns)}  (expected 274)')
print(f'Missing vals: {df_out.isnull().sum().sum()}')
print(f'\nColumn groups:')
print(f'  HSV/RGB (f000-f191)  : {sum(1 for c in df_out.columns if c.startswith("f") and c[1:].isdigit())}')
print(f'  Texture (tex_*)      : {sum(1 for c in df_out.columns if c.startswith("tex_"))}')
print(f'  Edge (f_edge_*/f_canny_*/...) : {sum(1 for c in df_out.columns if any(c.startswith(p) for p in ["f_edge","f_canny","f_sobel","f_laplacian","f_contour"]))}')
print(f'  HOG orientation (f_hog_*) : {sum(1 for c in df_out.columns if c.startswith("f_hog_"))}')
print(f'  HOG 9-bin (hog_bin_*) : {sum(1 for c in df_out.columns if c.startswith("hog_bin_"))}')

# ── Save ──────────────────────────────────────────────────────────────────────
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(OUTPUT_CSV, index=False)
print(f'\n✓ Saved → {OUTPUT_CSV}  ({OUTPUT_CSV.stat().st_size/1024**2:.1f} MB)')

## 6. Feature Visualizations

Three charts giving a clear picture of the extracted feature distributions — one per feature family.
These are computed from the saved `df_out` so they match exactly what landed in `cell_features.csv`.
